# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [3]:
DATA_PATH = "diabetes_prediction_dataset.csv"


df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (100000, 9)
polars shape: (100000, 9)

columns: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']

   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  

rows: 100,000
Size is fine.


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [4]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [5]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.013 ms   <- nothing was executed
lazy WITH .collect():       3.573 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.029]_ |
| Lazy query with `.collect()` | _[4.161]_ |

In one sentence — why is the first number meaningless?

_[Because without `.collect()`, Polars only creates the query plan without executing the actual operation.]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [6]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "gender"    # a text/category column, for grouping
NUMERIC_COL  = "age"     # a numeric column
NUMERIC_COL2 = "bmi"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] gender
[ok] age
[ok] bmi


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [7]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:    26.84 ms
polars:     3.30 ms
ratio:      8.13x


**Observation:** _[polars is faster than pandas at reading the CSV file, because polars uses multithreaded reader]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [8]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 43.0
pandas:     1.40 ms
polars:     0.32 ms
ratio:      4.44x


**Observation:** _[polars is faster because it uses multithreading and columnar memory, the indexing in pandas makes it slower]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [9]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.21 ms
polars:     0.12 ms
ratio:      1.84x


**Observation:** _[Polars is faster because it computes in parallel without copying the full data in memory]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [10]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:     3.43 ms
polars:     0.65 ms
ratio:      5.30x

pandas result (first rows):
gender
Female    2486310.6
Male      1701743.0
Other         532.0
Name: age, dtype: float64

polars result (first rows):
shape: (3, 2)
┌────────┬──────────┐
│ gender ┆ age      │
│ ---    ┆ ---      │
│ str    ┆ f64      │
╞════════╪══════════╡
│ Other  ┆ 532.0    │
│ Male   ┆ 1.7017e6 │
│ Female ┆ 2.4863e6 │
└────────┴──────────┘


**Observation:** _[polars is faster and handles aggregation much better than pandas]_

**Do both libraries give the same numbers?** _[yes but with a different row order]_

## 4.5 — Sorting and taking the top N

In [11]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:     7.28 ms
polars:     1.70 ms
ratio:      4.27x


**Observation:** _[polars is faster because it optimizes top-N queries instead of sorting everything ]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [12]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):    31.80 ms
polars (lazy + collect):     3.02 ms
ratio:    10.53x


**Look at the query plan.** Polars can show you what it decided to do.

In [13]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("age").sum()] BY [col("gender")]
  FROM
  Csv SCAN [diabetes_prediction_dataset.csv]
  PROJECT 2/9 COLUMNS
  SELECTION: col("age") > 43.0
  ESTIMATED ROWS: 100273


what does the plan show? 

it shows the optimized way that polars will take instead of running the code line by line

Can you see anything in it about which columns are read, or where the filter is
applied?

yes it shows that polars reads the needed columns and applies the filter early during the scan


## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [14]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:     5.50 ms
polars:     0.68 ms
ratio:      8.07x

rows before: 100000 -> after: 100000
row count unchanged: True


**Observation:** _[polars joins the tables faster because of the efficient memory management]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [15]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,26.84,3.30,8.13,polars
1,filter,1.40,0.32,4.44,polars
2,new column,0.21,0.12,1.84,polars
3,group by,3.43,0.65,5.30,polars
4,sort + top N,7.28,1.70,4.27,polars
5,chained pipeline,31.80,3.02,10.53,polars
6,join,5.50,0.68,8.07,polars


In [16]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   macOS-15.5-arm64-arm-64bit-Mach-O
   arm


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[The difference was largest in the chained pipeline operation, where Polars achieved a 10.53x speedup (3.02 ms vs 31.80 ms in Pandas), followed by read file with an 8.13x speedup. The difference was smallest in creating a new column, where Polars recorded a 1.84x speedup (0.12 ms vs 0.21 ms). These operations differ most because Polars optimizes multi-step pipelines and file parsing using multi-threading and query optimization in Rust, whereas single-column transformations involve minimal overhead for both libraries.]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[What surprised me most was seeing Polars outperform Pandas across all measured operations, achieving an impressive speedup of over 10x on complex chained queries and 8.13x on initial file loading on a 100,000-row dataset. Additionally, the efficiency of Polars during group by operations (5.30x speedup) demonstrated how well its parallel execution engine handles grouping compared to Pandas' single-threaded execution.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[Pandas syntax was initially easier to write because of familiar bracket indexing and standard method chaining. In contrast, Polars required learning its expression system with pl.col() and wrapping operations inside .select(), .with_columns(), or explicit .group_by().agg() contexts. However, once understood, Polars expressions proved much cleaner, less ambiguous, and more readable for complex transformations]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Yes, I reproduced and even exceeded the marketing claims on multiple operations. In our benchmark on the 100,000-row dataset, Polars achieved a 10.53x speedup in chained pipeline, 8.13x in read file, 8.07x in join, and 5.30x in group by. These extreme speedups occur because Polars runs operations in parallel across CPU cores with optimized memory layout, while Pandas executes sequentially on a single thread.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[I would choose Pandas for quick interactive data exploration, ad-hoc analysis on small-to-medium datasets, and workflows that depend heavily on standard machine learning packages like Scikit-Learn. I would choose Polars for large-scale data processing pipelines, ETL workflows, and performance-critical systems where parallel execution and query optimization significantly reduce runtime and memory usage.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[Library: DuckDB

Documentation: https://duckdb.org/docs/

Description: DuckDB is an in-process SQL OLAP database management system designed for fast analytical queries on tabular data. It allows users to execute high-performance SQL queries directly on Pandas DataFrames, Polars DataFrames, and CSV files without needing a separate database server.]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot